In [1]:
import numpy as np
from types import SimpleNamespace
from h5_utils import save, load

In [2]:
# Load all required data explicitly
RAW_DVL             = load('2_FILTERED_DVL.h5', 'DVL_FILTERED')  # Use filtered DVL data
RAW_ECHO            = load('1_RAW_DATA.h5', 'RAW_ECHO')
RAW_IMU_EULER       = load('1_RAW_DATA.h5', 'RAW_IMU_EULER')
RAW_IMU_QUAT        = load('1_RAW_DATA.h5', 'RAW_IMU_QUAT')
RAW_PROFILO_SEAKING = load('1_RAW_DATA.h5', 'RAW_PROFILO_SEAKING')
RAW_PROFILO_PING360 = load('1_RAW_DATA.h5', 'RAW_PROFILO_PING360')

Loaded DVL_FILTERED from 2_FILTERED_DVL.h5
Loaded RAW_ECHO from 1_RAW_DATA.h5
Loaded RAW_IMU_EULER from 1_RAW_DATA.h5
Loaded RAW_IMU_QUAT from 1_RAW_DATA.h5
Loaded RAW_PROFILO_SEAKING from 1_RAW_DATA.h5
Loaded RAW_PROFILO_PING360 from 1_RAW_DATA.h5


<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; margin: 25px 0 15px 0; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box; font-weight: 800; font-size: 20px;">Determination of the common time window of all sensors</h3>

</div>

In [3]:
# 3) We find the LATEST start time among all sensors...
t_init = max(
    RAW_DVL.ABS_TIME[0],
    RAW_ECHO.ABS_TIME[0],
    RAW_IMU_EULER.ABS_TIME[0],
    RAW_IMU_QUAT.ABS_TIME[0],
    RAW_PROFILO_SEAKING.ABS_TIME[0],
    RAW_PROFILO_PING360.ABS_TIME[0]
)

# 4) ...and the EARLIEST end time among all sensors.
t_end = min(
    RAW_DVL.ABS_TIME[-1],
    RAW_ECHO.ABS_TIME[-1],
    RAW_IMU_EULER.ABS_TIME[-1],
    RAW_IMU_QUAT.ABS_TIME[-1],
    RAW_PROFILO_SEAKING.ABS_TIME[-1],
    RAW_PROFILO_PING360.ABS_TIME[-1]
)

I_START = np.searchsorted(RAW_DVL.ABS_TIME, t_init, side='left')        # We locate the first position = index where ABS_TIME >= t_init ...
I_END   = np.searchsorted(RAW_DVL.ABS_TIME, t_end , side='right') - 1   # ... and the first position = index where ABS_TIME > t_end, minus 1 ⇒ last position where <= t_end

#I_START = np.where(RAW_DVL.ABS_TIME >= t_init)[0][0] # “First index where condition is True”   # np.where(cond)[0] returns an array of all matching indices, and then [0] or [-1] picks the first or last one.
#I_END   = np.where(RAW_DVL.ABS_TIME <= t_end)[0][-1] # “Last index where condition is True”

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; margin: 25px 0 15px 0; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box; font-weight: 800; font-size: 20px;">We create DVL_VALID. which contains only the samples in the shared time window, with ABS_TIME re-zeroed at t_init.</h3>

</div>

In [4]:
DVL_VALID = SimpleNamespace()
for field, array in vars(RAW_DVL).items():
    window = array[I_START:I_END + 1]
    if field == 'ABS_TIME':
        setattr(DVL_VALID, field, window - t_init) # We subtract the t_init from every element of the sliced timestamp array, so the first valid time becomes 0.
    else:
        setattr(DVL_VALID, field, window)

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; margin: 25px 0 15px 0; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box; font-weight: 800; font-size: 20px;">We align Echosounder data to the common time window by creating ECHO_VALID</h3>

</div>

In [5]:
I_START = np.searchsorted(RAW_ECHO.ABS_TIME, t_init, side='left')
I_END   = np.searchsorted(RAW_ECHO.ABS_TIME, t_end, side='right') - 1

ECHO_VALID = SimpleNamespace()

ECHO_VALID.ABS_TIME     = RAW_ECHO.ABS_TIME[I_START : I_END + 1] - t_init
ECHO_VALID.DISTANCE     = RAW_ECHO.DISTANCE[I_START : I_END + 1]
ECHO_VALID.WATER_TEMPER = RAW_ECHO.WATER_TEMPER[I_START : I_END + 1]
ECHO_VALID.QUAT         = RAW_ECHO.QUAT[I_START : I_END + 1, :]

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; margin: 25px 0 15px 0; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box; font-weight: 800;font-size: 20px;">We align IMU Euler-angle and quaternion data to the common time window</h3>

</div>

In [6]:
I_START = np.searchsorted(RAW_IMU_EULER.ABS_TIME, t_init, side='left')
I_END   = np.searchsorted(RAW_IMU_EULER.ABS_TIME, t_end,   side='right') - 1

IMU_VALID_EULER = SimpleNamespace()

IMU_VALID_EULER.ABS_TIME     = RAW_IMU_EULER.ABS_TIME[I_START : I_END + 1] - t_init
IMU_VALID_EULER.EULER_ANGLE  = RAW_IMU_EULER.EULER_ANGLE[I_START : I_END + 1, :]

# --------------------------------------------------------------------------

I_START = np.searchsorted(RAW_IMU_QUAT.ABS_TIME, t_init, side='left')
I_END   = np.searchsorted(RAW_IMU_QUAT.ABS_TIME, t_end,   side='right') - 1

IMU_VALID_QUAT = SimpleNamespace()

IMU_VALID_QUAT.ABS_TIME = RAW_IMU_QUAT.ABS_TIME[I_START : I_END + 1] - t_init
IMU_VALID_QUAT.QUAT     = RAW_IMU_QUAT.QUAT[I_START : I_END + 1, :]

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; margin: 25px 0 15px 0; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box; font-weight: 800; font-size: 20px;">We align Seaking vertical profiling data to the common time window</h3>

</div>

In [7]:
I_START = np.searchsorted(RAW_PROFILO_SEAKING.ABS_TIME, t_init, side='left')
I_END   = np.searchsorted(RAW_PROFILO_SEAKING.ABS_TIME, t_end,   side='right') - 1

PROFILO_SEAKING_VALID = SimpleNamespace()

PROFILO_SEAKING_VALID.ABS_TIME = (RAW_PROFILO_SEAKING.ABS_TIME[I_START : I_END + 1] - t_init)
PROFILO_SEAKING_VALID.ANGLE    = (RAW_PROFILO_SEAKING.ANGLE[I_START : I_END + 1])
PROFILO_SEAKING_VALID.RANGE    = (RAW_PROFILO_SEAKING.RANGE[I_START : I_END + 1])
PROFILO_SEAKING_VALID.NBR_BINS = (RAW_PROFILO_SEAKING.NBR_BINS[I_START : I_END + 1])
PROFILO_SEAKING_VALID.SKYLINE  = (RAW_PROFILO_SEAKING.SKYLINE[I_START : I_END + 1, :])

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden; font-size: 28px;">

<h3 style="color: #5d4e37; margin: 25px 0 15px 0; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box; font-weight: 800; font-size: 20px;">We align Ping360 horizontal profiling data to the common time window</h3>

</div>

In [8]:
I_START = np.searchsorted(RAW_PROFILO_PING360.ABS_TIME, t_init, side='left')
I_END   = np.searchsorted(RAW_PROFILO_PING360.ABS_TIME, t_end,   side='right') - 1

PROFILO_PING360_VALID = SimpleNamespace()

PROFILO_PING360_VALID.ABS_TIME  = (RAW_PROFILO_PING360.ABS_TIME[I_START : I_END + 1] - t_init)
PROFILO_PING360_VALID.MODE      = RAW_PROFILO_PING360.MODE[I_START : I_END + 1]
PROFILO_PING360_VALID.GAIN      = RAW_PROFILO_PING360.GAIN[I_START : I_END + 1]
PROFILO_PING360_VALID.ANGLE     = RAW_PROFILO_PING360.ANGLE[I_START : I_END + 1]
PROFILO_PING360_VALID.NBR_SCANS = RAW_PROFILO_PING360.NBR_SCANS[I_START : I_END + 1]
PROFILO_PING360_VALID.NBR_BINS  = RAW_PROFILO_PING360.NBR_BINS[I_START : I_END + 1]
PROFILO_PING360_VALID.SKYLINE   = RAW_PROFILO_PING360.SKYLINE[I_START : I_END + 1, :]

In [9]:
save('HDF5_DATA/3_VALID_DATA.h5', 'DVL_VALID', DVL_VALID, mode='w')
save('HDF5_DATA/3_VALID_DATA.h5', 'ECHO_VALID', ECHO_VALID)
save('HDF5_DATA/3_VALID_DATA.h5', 'IMU_VALID_EULER', IMU_VALID_EULER)
save('HDF5_DATA/3_VALID_DATA.h5', 'IMU_VALID_QUAT', IMU_VALID_QUAT)
save('HDF5_DATA/3_VALID_DATA.h5', 'PROFILO_SEAKING_VALID', PROFILO_SEAKING_VALID)
save('HDF5_DATA/3_VALID_DATA.h5', 'PROFILO_PING360_VALID', PROFILO_PING360_VALID)

Saved to HDF5_DATA/3_VALID_DATA.h5 -> DVL_VALID
Saved to HDF5_DATA/3_VALID_DATA.h5 -> ECHO_VALID
Saved to HDF5_DATA/3_VALID_DATA.h5 -> IMU_VALID_EULER
Saved to HDF5_DATA/3_VALID_DATA.h5 -> IMU_VALID_QUAT
Saved to HDF5_DATA/3_VALID_DATA.h5 -> PROFILO_SEAKING_VALID
Saved to HDF5_DATA/3_VALID_DATA.h5 -> PROFILO_PING360_VALID
